In [ ]:
!pip install -q -U groq

from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

def ask(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-20b"
    )
    return response.choices[0].message.content

print("✅ Ready!")

✅ Ready!


## 📖 Technique 1: Zero-Shot Prompting

Direct question without examples.

In [ ]:
# Zero-shot: Just ask directly
prompt = "Classify this review as positive or negative: 'The food was terrible and cold.'"

print(ask(prompt))

Negative


## 📖 Technique 2: Few-Shot Prompting

Provide examples to guide the model.

In [ ]:
# Few-shot: Give examples first
prompt = """
Classify the sentiment:

Review: "Amazing service, loved it!" -> Positive
Review: "Worst experience ever" -> Negative
Review: "It was okay, nothing special" -> Neutral

Review: "The product exceeded my expectations!" ->
"""
print(ask(prompt))

Review: "The product exceeded my expectations!" -> Positive


## 📖 Technique 3: Chain-of-Thought (CoT)

Ask the model to think step by step.

In [ ]:
# Without CoT
print("Without CoT:")
print(ask("What is 23 * 17?"))

# With CoT
print("\nWith CoT:")
cot_prompt = """
What is 23 * 17?

Let's solve this step by step:
"""
print(ask(cot_prompt))

Without CoT:
23 × 17 = 391

With CoT:
Sure! Let’s break it down into simple steps:

---

### 1. Write the problem in expanded form

\(23 \times 17\)

---

### 2. Break each number into its place‑value components

- \(23 = 20 + 3\)
- \(17 = 10 + 7\)

---

### 3. Use the distributive property (FOIL method)

Multiply each part of the first number by each part of the second number:

| Multiply | Result |
|----------|--------|
| \(20 \times 10\) | \(200\) |
| \(20 \times 7\)  | \(140\) |
| \(3 \times 10\)  | \(30\)  |
| \(3 \times 7\)   | \(21\)  |

---

### 4. Add all the partial products

\[
200 + 140 + 30 + 21 = 391
\]

---

### 5. Final answer

\[
23 \times 17 = \boxed{391}
\]

If you’d like, you can double‑check by using standard long multiplication:

```
      23
    × 17
    -----
     161   ← 23 × 7
    230    ← 23 × 10 (shifted one place to the left)
    -----
     391
```

Both methods give the same result: **391**.


## 📖 Technique 4: Role-Based Prompting

In [ ]:
# Assign a role to the AI
prompt = """
You are an expert Python programmer with 20 years of experience.
You explain code simply and always provide examples.

Explain what a list comprehension is.
"""
print(ask(prompt))

## What is a List Comprehension?

A **list comprehension** is a compact syntax that lets you build a new list from an existing iterable (list, tuple, string, etc.) in a single, readable line of code.

Think of it as a *mini‑loop* that:

1. **Iterates** over each element in the source iterable.
2. **Computes** a value (the expression part) for each element.
3. **(Optionally) Filters** out elements that don’t satisfy a condition.

The result is a **new list** that contains the computed values.

---

### Basic Anatomy

```python
[ <expression> for <item> in <iterable> if <condition> ]
```

| Part | What it does | Example |
|------|--------------|---------|
| `<expression>` | The value you want in the new list | `x ** 2` |
| `<item>` | A loop variable that takes each element | `x` |
| `<iterable>` | The source you iterate over | `range(10)` |
| `<condition>` (optional) | A filter that decides whether to keep the element | `x % 2 == 0` |

---

## Simple Example

**Goal**: Create a list of s

## 📖 Technique 5: Structured Output

In [ ]:
# Request specific format
prompt = """
Analyze this product review and respond in JSON format:

Review: "Great laptop, fast performance, but battery life could be better. Worth the price."

Respond with this exact JSON structure:
{
  "sentiment": "positive/negative/mixed",
  "pros": ["list of pros"],
  "cons": ["list of cons"],
  "rating": "1-5"
}
"""
print(ask(prompt))

{
  "sentiment": "positive",
  "pros": [
    "Great laptop",
    "fast performance",
    "worth the price"
  ],
  "cons": [
    "battery life could be better"
  ],
  "rating": "4"
}


## 💻 Lab: Build a Prompt Library

In [ ]:
# Create a reusable prompt library
class PromptLibrary:

    @staticmethod
    def summarize(text, length="short"):
        return f"Summarize this text in a {length} paragraph:\n\n{text}"

    @staticmethod
    def translate(text, target_lang):
        return f"Translate to {target_lang}:\n\n{text}"

    @staticmethod
    def code_explain(code):
        return f"Explain this code line by line:\n\n```\n{code}\n```"

    @staticmethod
    def sentiment(text):
        return f"""Analyze sentiment. Respond with JSON:
{{"sentiment": "positive/negative/neutral", "confidence": "high/medium/low", "reason": "brief reason"}}

Text: {text}"""

    @staticmethod
    def quiz_question(topic, difficulty="medium"):
        return f"""Create a {difficulty} difficulty quiz question about {topic}.
Format:
Question: [question]
A) [option]
B) [option]
C) [option]
D) [option]
Answer: [letter]"""

# Test the library
pl = PromptLibrary()
print(ask(pl.quiz_question("Python loops", "easy")))

Question: Which loop type is used to iterate over a sequence of values in Python?

A) if  
B) for  
C) switch  
D) return  

Answer: B


## 🛠️ Mini Project: Prompt Template System

In [ ]:
class PromptTemplate:
    def __init__(self, template):
        self.template = template

    def format(self, **kwargs):
        return self.template.format(**kwargs)

    def run(self, **kwargs):
        prompt = self.format(**kwargs)
        return ask(prompt)

# Create templates
email_template = PromptTemplate("""
Write a {tone} email about {topic}.
- Keep it {length}
- Include a clear call to action
""")

# Use the template
result = email_template.run(
    tone="professional",
    topic="project deadline extension",
    length="brief (3-4 sentences)"
)
print(result)

**Subject:** Request for Extension of Project Deadline – [Project Name]  

Dear [Manager’s Name],  

I am writing to request a brief extension of the deadline for [Project Name] from **[original date]** to **[new date]** due to unforeseen scope adjustments. This additional time will allow us to finalize all deliverables to the required quality standard. Please confirm whether this extension is acceptable by replying to this email by **[date]**.  

Thank you for your understanding.  

Best regards,  
[Your Name]  
